### 이전 코드 + 궤적 애니메이션 시각화

In [2]:
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# state(상태) - 11개의 이산적인 구간으로 나눠서 표현
state_space = np.linspace(-1.0, 1.0, 11)
print(state_space)

# 행동 구간 정의
action_space = [-1, 0, 1]
# Q[state_index, action_index]
q_table = np.zeros((len(state_space), len(action_space)))
print(q_table)

# 하이퍼 파라미터
alpha = 0.1
gammer = 0.9
epsilon = 0.9
epsilon_decay = 0.995
epsilon_min = 0.01

episodes = 500

# 현재 위치를 이산화하여 상태 인덱스로 변환 후 반환
def get_state_index(position):
  return np.argmin(np.abs(state_space - position))
print(get_state_index(-0.2), ' ',state_space[get_state_index(-0.2)])

# 보상(reward)
def get_reward(position):
  return -abs(position) # 중심을 벗어나는 모든 값들 전부 패널티 : Negative Reward
print(get_reward(0.5))

# 환경 동작 관련 정의
# 현재 위치에서 어떤 행동을 했을 때 '새로운 위치와 보상을 계산'
def stepFunc(position, action):
  position += action * 0.1
  position = np.clip(position, -1.0, 1.0)
  reward = get_reward(position) # 이동 후 보상위치 반환
  return position, reward

[-1.  -0.8 -0.6 -0.4 -0.2  0.   0.2  0.4  0.6  0.8  1. ]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
4   -0.19999999999999996
-0.5


In [6]:
reward_list = []
trajectories = [] # 에이전트 궤적 저장

# 학습
for ep in range(episodes):
  # 매 에피소드 마다 에이전트는 임의의 위치에서 출발
  position = np.random.uniform(-1.0, 1.0) # position의 위치를 균등분포로 줌
  total_reward = 0
  trajectory= [] # 각 에피소드 마다 이동 기록

  for _ in range(50):
    state_index = get_state_index(position) # q_table을 사용 할 준비

    #
    if random.random() < epsilon:
      action_idx = random.choice([0, 1, 2])
    else:
      action_idx = np.argmax(q_table[state_index])

    # 행동 선택
    action = action_space[action_idx]
    next_position , reward = stepFunc(position, action)

    # 다음 위치에 대한 이산 상태 인덱스 계산
    next_state_index = get_state_index(next_position)

    # 다음 상태 에서 선택 가능한 가장 높은 q값 - q값 갱신
    best_next_q = np.max(q_table[next_state_index])

    # Q 테이블 갱신
    q_table[state_index, action_idx] += alpha * (reward + gammer * best_next_q - q_table[state_index, action_idx])
    position = next_position
    total_reward += reward
    trajectory.append(next_position)

  reward_list.append(total_reward) # 에피소드 마다 총 보상 기록
  epsilon = max(epsilon_min, epsilon * epsilon_decay)

  if ep % 10 == 0: # 10 에피소드 마다 궤적 저장
    trajectories.append(trajectory)

  epsilon = max(0.05, epsilon * epsilon_decay)

# 궤적 평탄화 작업
flat_positions = [pos for traj in trajectories for pos in traj]
print("flat_positions :", flat_positions)

frame_count = len(flat_positions)
print("frame_count :", frame_count)

flat_positions : [np.float64(0.8036092489187737), np.float64(0.7036092489187737), np.float64(0.7036092489187737), np.float64(0.6036092489187738), np.float64(0.5036092489187738), np.float64(0.4036092489187738), np.float64(0.30360924891877383), np.float64(0.20360924891877383), np.float64(0.10360924891877382), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.10360924891877382), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.0036092489187738164), np.float64(0.003609248918773

In [15]:
from IPython.utils import frame
from matplotlib import transforms

fig, ax = plt.subplots()
ax.set_xlim(-1.1, 1.0)
ax.set_ylim(-0.1, 1.05)
ax.set_xlabel('Position on road (-1.0 ~ 1.0)')
ax.axvline(0, color='magenta', linestyle='--', label='Center Line')
ax.set_title("Agent Simulation (Q-learning)")
point, = ax.plot([], [], 'bo', markersize=8)
ax.legend()

# 에피소드 번호 화면에 표시
episode_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)

# 애니메이션의 정의 : 프레임 수 만큼 호출되며, 각 프레임에서 파란점을 갱신
def updateFunc(frame):
  x = flat_positions[frame] # 현재 프레임의 에이전트 위치
  y = (frame % 50) / 50 # 정규화된 스텝 위치(y축)
  point.set_data([x],[y])
  episode_num = frame // 50 + 1
  episode_text.set_text(f'Episode : {episode_num}')

ani = FuncAnimation(fig, updateFunc, frames=frame_count, interval=100, repeat=False)
plt.close(fig)

HTML(ani.to_jshtml())

Output hidden; open in https://colab.research.google.com to view.

In [16]:
# 애니메이션 저장하기(mp4)
ani.save('ani_car.mp4', writer='ffmpeg', fps=10)

from google.colab import files
files.download('ani_car.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
# Q_tabel 출력
print('\nfinal Q-tabel (rows=states, columns=actions[-1, 0, 1])\n')
# >7 : 오른쪽 자리 7자리 정렬
header = f'{'satate':>7} | {'-1':>7}{'0':>7}{'1':>7}'
print(header)
print('-' * len(header))
for i, state_val in enumerate(state_space):
  q_vals = q_table[i]
  print(f'{state_val:>7.2f} | {q_vals[0]:>7.3f}{q_vals[1]:>7.3f}{q_vals[2]:>7.3f}')



final Q-tabel (rows=states, columns=actions[-1, 0, 1])

 satate |      -1      0      1
-------------------------------
  -1.00 |  -3.123 -3.058 -3.007
  -0.80 |  -2.923 -2.835 -2.431
  -0.60 |  -2.421 -1.996 -1.563
  -0.40 |  -1.578 -1.214 -0.918
  -0.20 |  -0.779 -0.650 -0.535
   0.00 |  -0.588 -0.506 -0.595
   0.20 |  -0.545 -0.632 -0.874
   0.40 |  -0.940 -1.232 -1.631
   0.60 |  -1.647 -1.918 -2.204
   0.80 |  -2.382 -2.782 -3.024
   1.00 |  -2.952 -3.148 -3.120
